In [1]:
# 安裝所有必要的套件
print("正在安裝所有必要的套件...")
!pip install yt-dlp -q
!pip install openai-whisper -q
!pip install langchain langchain-community langchain-core -q # 安裝 LangChain 相關套件
!pip install langchain-ollama -q # 安裝 Ollama 的 LangChain 整合套件
print("✅ 所有套件安裝完成！")

正在安裝所有必要的套件...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 174.3/174.3 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 73.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 800.5/800.5 kB 33.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 106.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 78.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 62.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/12

In [2]:
# 導入套件
import subprocess
import os
import time
import whisper
from langchain_community.llms import Ollama # 從 langchain-community 導入 Ollama

In [3]:
# 定義主要函式
def setup_and_run_ollama():
    """
    設定並啟動 Ollama 伺服器，拉取模型，並返回一個 LangChain LLM 物件。
    """
    print("--- 步驟 1：設定並啟動本地 LLM (Ollama) ---")
    # 安裝 Ollama
    print("正在安裝 Ollama...")
    process = subprocess.run('curl -fsSL https://ollama.com/install.sh | sh', shell=True, capture_output=True, text=True)
    if process.returncode != 0:
        print("Ollama 安裝失敗。")
        print(process.stderr)
        return None

    # 在背景啟動 Ollama 伺服器
    print("正在背景啟動 Ollama 伺服器...")
    os.system('ollama serve > server.log 2>&1 &')
    # 等待幾秒鐘確保伺服器有足夠時間啟動
    time.sleep(5)

    # 拉取 gemma:7b 模型 (如果模型已存在，會快速跳過)
    print("正在拉取 gemma:7b 模型 (如果已下載則會跳過)...")
    os.system('ollama pull gemma:7b > model_pull.log 2>&1 &')

    # 等待模型拉取完成，這個過程可能需要幾分鐘
    # 我們透過檢查日誌來判斷是否完成
    pull_finished = False
    for i in range(180): # 最多等待 3 分鐘
        time.sleep(1)
        if 'success' in open('model_pull.log').read():
            print("✅ 模型拉取成功！")
            pull_finished = True
            break
        elif 'already exists' in open('model_pull.log').read():
            print("✅ 模型已存在，無需下載。")
            pull_finished = True
            break
    if not pull_finished:
        print("模型拉取超時或失敗，請檢查 'model_pull.log'。")
        return None

    # 初始化 LangChain 的 Ollama LLM 物件
    print("正在初始化 LangChain LLM 物件...")
    try:
        llm = Ollama(model="gemma:7b")
        # 測試連線
        llm.invoke("Hi")
        print("✅ LLM (gemma:7b on Ollama) 連線成功！")
        return llm
    except Exception as e:
        print(f"連接到 Ollama LLM 時發生錯誤：{e}")
        print("請檢查 'server.log' 確認伺服器狀態。")
        return None

In [4]:
def get_youtube_url():
    """
    提示使用者輸入 YouTube 影片 URL。
    """
    print("\n--- 步驟 2：輸入 YouTube 網址 ---")
    url = input("請輸入 YouTube 影片的 URL：")
    return url

def download_mp3(url, output_filename="downloaded_audio.mp3"):
    """
    使用 yt-dlp 下載影片的音訊。
    """
    print("\n--- 步驟 3：下載音訊 ---")
    if os.path.exists(output_filename):
        os.remove(output_filename)
    command = f"yt-dlp -x --audio-format mp3 -o '{output_filename}' {url}"
    print(f"正在下載 MP3 檔案，儲存為 {output_filename}...")
    try:
        subprocess.run(command, shell=True, check=True, capture_output=True, text=True)
        print("✅ 音訊下載完成！")
        return output_filename
    except subprocess.CalledProcessError as e:
        print(f"下載音訊時發生錯誤：\n{e.stderr}")
        return None

def transcribe_audio(audio_filename):
    """
    使用 Whisper 辨識音訊，並回傳逐字稿和偵測到的語言。
    """
    print("\n--- 步驟 4：語音轉文字 (Whisper) ---")
    print("正在使用 Whisper 辨識語音內容，這可能需要幾分鐘時間...")
    model = whisper.load_model("medium")
    result = model.transcribe(audio_filename, fp16=False)
    detected_language = result['language']
    transcript_text = result['text']
    print(f"✅ 語音辨識完成！偵測到的語言：{detected_language}")
    print("\n【原始逐字稿】:")
    print(transcript_text)
    return transcript_text, detected_language

In [5]:
def process_transcript_with_llm(transcript, language, llm):
    """
    根據語言，使用 Ollama LLM 處理逐字稿 (加標點或翻譯)。
    """
    print("\n--- 步驟 5：文字處理 (Ollama LLM) ---")
    if language in ['zh', 'ja', 'yue']: # 中文、日文、粵語都視為中文處理
        print("偵測到中文逐字稿，正在請 LLM 加上標點符號與分段...")
        prompt = f"你是一位專業的中文文稿編輯。請為以下由語音辨識產出的中文逐字稿，加上完整且符合台灣慣用語氣的標點符號，並進行適當的分段，讓它成為一篇通順易讀的文章。請直接輸出整理好的文章，不要包含任何前言或結語。逐字稿如下：\n\n{transcript}"
    else:
        print(f"偵測到 {language} 逐字稿，正在請 LLM 翻譯成繁體中文...")
        prompt = f"你是一位專業的翻譯師。請將以下英文逐字稿，精確且流暢地翻譯成繁體中文。請直接輸出翻譯後的中文內容，不要添加任何額外的解釋或說明。英文逐字稿如下：\n\n{transcript}"

    try:
        response = llm.invoke(prompt)
        print("✅ LLM 處理完成！")
        return response
    except Exception as e:
        print(f"在請求 LLM 處理文字時發生錯誤: {e}")
        return "LLM 處理失敗。"

def summarize_into_bullet_points(text, llm):
    """
    使用 Ollama LLM 將處理過的文字摘要成條列式重點。
    """
    print("\n--- 步驟 6：內容摘要 (Ollama LLM) ---")
    print("正在請 LLM 將內容摘要成條列式重點...")
    prompt = f"你是一位專業的內容分析師。請將以下文章內容，整理成一份清晰的重點摘要，並以條列式（使用「-」或「*」）呈現。摘要需涵蓋文章的主要觀點與核心資訊。文章內容如下：\n\n{text}"
    try:
        response = llm.invoke(prompt)
        print("✅ 摘要完成！")
        return response
    except Exception as e:
        print(f"在請求 LLM 摘要時發生錯誤: {e}")
        return "LLM 摘要失敗。"

In [ ]:
# 執行主程式
# 啟動 Ollama 並取得 LLM 物件
llm = setup_and_run_ollama()
if not llm:
  print("LLM 初始化失敗，程式終止。")

# 執行後續流程
url = get_youtube_url()
mp3_file = download_mp3(url)

transcript, language = transcribe_audio(mp3_file)
if not transcript:
  os.remove(mp3_file)

processed_text = process_transcript_with_llm(transcript, language, llm)
print("\n【LLM 處理後結果 (翻譯/標點)】:")
print(processed_text)

summary = summarize_into_bullet_points(processed_text, llm)
print("\n【LLM 條列式重點摘要】:")
print(summary)

# 清理檔案
os.remove(mp3_file)
print(f"\n✅ 任務完成！已刪除暫存檔案：{mp3_file}")

--- 步驟 1：設定並啟動本地 LLM (Ollama) ---
正在安裝 Ollama...
正在背景啟動 Ollama 伺服器...
正在拉取 gemma:7b 模型 (如果已下載則會跳過)...
✅ 模型拉取成功！
正在初始化 LangChain LLM 物件...


<ipython-input-3-2966792742>:45: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaLLM``.
  llm = Ollama(model="gemma:7b")


✅ LLM (gemma:7b on Ollama) 連線成功！

--- 步驟 2：輸入 YouTube 網址 ---
請輸入 YouTube 影片的 URL：https://www.youtube.com/watch?v=fVAMKbBGzTY

--- 步驟 3：下載音訊 ---
正在下載 MP3 檔案，儲存為 downloaded_audio.mp3...
✅ 音訊下載完成！

--- 步驟 4：語音轉文字 (Whisper) ---
正在使用 Whisper 辨識語音內容，這可能需要幾分鐘時間...
✅ 語音辨識完成！偵測到的語言：en

【原始逐字稿】:
 Here we're going to look at approximation and estimation. A skill that will be required for this is rounding, so if you need a refresher, check out our video here. An approximation is anything that is similar but not exactly the same as something else. For example, if you were to say a 57 minute journey would take about an hour, you would be approximating. A value can be approximated by rounding, usually to a value that is easier to work with. Be aware, for approximation questions you are rarely allowed a calculator. A good rule of thumb for approximating is to round to a position that is one or two positions above the lowest position of the original value. Let's see an example of this rule in action. S